<a href="https://colab.research.google.com/github/nurin1732/Blood-Prediction-Project/blob/main/preprocessingAndCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import pandas as pd
import numpy as np

In [28]:
#---------------
# LOAD DATA
#---------------

# Define the raw data URL from MoH GitHub
url = "https://raw.githubusercontent.com/MoH-Malaysia/data-darah-public/main/donations_state.csv"

# Load the dataset into a Pandas DataFrame
df_raw = pd.read_csv(url)

# Copy into working dataframe
df = df_raw.copy()

print(df.head())

         date     state  daily  blood_a  blood_b  blood_o  blood_ab  \
0  2006-01-01  Malaysia    525      152      139      194        40   
1  2006-01-02  Malaysia    227       53       43      112        19   
2  2006-01-03  Malaysia    112       29       21       56         6   
3  2006-01-04  Malaysia    391       92       98      165        36   
4  2006-01-05  Malaysia    582      149      198      193        42   

   location_centre  location_mobile  type_wholeblood  type_apheresis_platelet  \
0              308              217              525                        0   
1              162               65              217                        6   
2              112                0               89                       10   
3              145              246              371                        4   
4              371              211              548                       17   

   type_apheresis_plasma  type_other  social_civilian  social_student  \
0            

In [29]:
#--------------------
# PRE-PROCESSING
#--------------------

# Convert date to datetime objects
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Remove invalid dates
df = df.dropna(subset=['date'])

#--------------------
# SELECT COLUMNS
#--------------------

selected_columns = [
    'date','state','daily',
    'blood_a','blood_b','blood_ab','blood_o',
    'donations_new','donations_regular','donations_irregular'
]

df = df[selected_columns]

print(df.head())

        date     state  daily  blood_a  blood_b  blood_ab  blood_o  \
0 2006-01-01  Malaysia    525      152      139        40      194   
1 2006-01-02  Malaysia    227       53       43        19      112   
2 2006-01-03  Malaysia    112       29       21         6       56   
3 2006-01-04  Malaysia    391       92       98        36      165   
4 2006-01-05  Malaysia    582      149      198        42      193   

   donations_new  donations_regular  donations_irregular  
0            243                277                    5  
1             83                143                    1  
2              8                101                    3  
3            286                102                    3  
4            328                250                    4  


In [30]:
#------------------
# DATA CLEANING
#------------------

# Remove duplicates
df = df.drop_duplicates()

# Remove Malaysia total row
df = df[df['state'] != 'Malaysia'].copy()

# Handle missing values
df = df.fillna(0)

print("Unique states:")
print(df['state'].unique())

Unique states:
['Johor' 'Kedah' 'Kelantan' 'Melaka' 'Negeri Sembilan' 'Pahang' 'Perak'
 'Pulau Pinang' 'Sabah' 'Sarawak' 'Selangor' 'Terengganu'
 'W.P. Kuala Lumpur']


In [31]:
#----------
# IQR
#----------

df['spike_flag'] = 'none'

# Calculate IQR bounds for each state
Q1 = df.groupby('state')['daily'].transform(
    lambda x: x.quantile(0.25)
)

Q3 = df.groupby('state')['daily'].transform(
    lambda x: x.quantile(0.75)
)

IQR = Q3 - Q1

normal_upper = Q3 + 1.5 * IQR
extreme_upper = Q3 + 3.0 * IQR

# Normal spikes
df.loc[
    (df['daily'] > normal_upper) &
    (df['daily'] <= extreme_upper),
    'spike_flag'
] = 'normal_spike'

# Extreme spikes
df.loc[
    df['daily'] > extreme_upper,
    'spike_flag'
] = 'extreme_spike'

print("--- Flag Counts ---")
print(df['spike_flag'].value_counts())

--- Flag Counts ---
spike_flag
none             93320
normal_spike      3098
extreme_spike      588
Name: count, dtype: int64


In [32]:
#----------------------
# FILTER (2023-2025)
#----------------------

# Filtering for post-pandemic data
df = df[
    (df['date'].dt.year >= 2023) &
    (df['date'].dt.year <= 2025)
].copy()

print(df['date'].min(), "to", df['date'].max())

2023-01-01 00:00:00 to 2025-12-31 00:00:00


In [33]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/nurin1732/Blood-Prediction-Project/refs/heads/main/model/data/Training(2023-2024).csv"
test_url = "https://raw.githubusercontent.com/nurin1732/Blood-Prediction-Project/refs/heads/main/model/data/Testing(2025).csv"

train_holidays = pd.read_csv(train_url)
test_holidays = pd.read_csv(test_url)

In [34]:
import pandas as pd
import numpy as np

#----------------------------------
# HOLIDAY FEATURE ENGINEERING
#----------------------------------


# Combine both holiday files
holiday_df = pd.concat([train_holidays, test_holidays])

# Convert holiday dates to datetime
holiday_df['date'] = pd.to_datetime(
    holiday_df['date'],
    errors='coerce'
)

# is_holiday
df['is_holiday'] = (
    df['date'].isin(holiday_df['date'])
).astype(int)

# is_weekend (state-based weekends)
def classify_weekend(row):
    state = row['state']
    day = row['date'].dayofweek # 0=Mon, 4=Fri, 5=Sat, 6=Sun

    # 1. Define the special states
    fri_sat_states = ['Kedah', 'Kelantan', 'Terengganu']

    # 2. Logic for Fri/Sat states
    if state in fri_sat_states:
        return 1 if day in [4, 5] else 0 # Only 4 (Fri) and 5 (Sat) are 1

    # 3. Logic for all other states
    else:
        return 1 if day in [5, 6] else 0 # Only 5 (Sat) and 6 (Sun) are 1

# Apply the logic
df['is_weekend'] = df.apply(classify_weekend, axis=1)

# Days to next holiday
holiday_dates = holiday_df['date'].sort_values()

def days_until_next_holiday(current_date):

    future = holiday_dates[holiday_dates >= current_date]

    if len(future) == 0:
        return np.nan

    next_holiday = future.iloc[0]

    return (next_holiday - current_date).days

df['days_to_holiday'] = df['date'].apply(
    days_until_next_holiday
)

print(df.head())

            date  state  daily  blood_a  blood_b  blood_ab  blood_o  \
13671 2023-01-01  Johor    346       91       81        18      156   
13672 2023-01-02  Johor    174       43       38         8       85   
13673 2023-01-03  Johor     88       18       38         0       32   
13674 2023-01-04  Johor    122       28       35         8       51   
13675 2023-01-05  Johor     60       20       15         5       20   

       donations_new  donations_regular  donations_irregular    spike_flag  \
13671             67                210                   69  normal_spike   
13672             39                105                   30          none   
13673             21                 48                   19          none   
13674             62                 41                   19          none   
13675              8                 41                   11          none   

       is_holiday  is_weekend  days_to_holiday  
13671           0           1             21.0  
13672 

In [35]:
#-------------------------------------------
# DONOR RETENTION FEATURES
#-------------------------------------------

# Ratio of regular donors to total donors
df['regular_ratio'] = ((
    df['donations_regular'] / (df['daily'] + 1)
).round(2)
)

# Sort by state and date
df = df.sort_values(['state', 'date'])

# 7-day rolling average for regular donors
df['7dayave_regular'] = (
    df.groupby('state')['donations_regular'].transform(
        lambda x:
        x.shift(1).rolling(
            window=7,
            min_periods=1
        ).mean()
    ).round(2).fillna(0)
)

# 30-day rolling average for new donors
df['30dayave_new'] = (
    df.groupby('state')['donations_new'].transform(
        lambda x:
        x.shift(1).rolling(
            window=30,
            min_periods=1
        ).mean()
    ).round(2).fillna(0)
)

print(df.head())

            date  state  daily  blood_a  blood_b  blood_ab  blood_o  \
13671 2023-01-01  Johor    346       91       81        18      156   
13672 2023-01-02  Johor    174       43       38         8       85   
13673 2023-01-03  Johor     88       18       38         0       32   
13674 2023-01-04  Johor    122       28       35         8       51   
13675 2023-01-05  Johor     60       20       15         5       20   

       donations_new  donations_regular  donations_irregular    spike_flag  \
13671             67                210                   69  normal_spike   
13672             39                105                   30          none   
13673             21                 48                   19          none   
13674             62                 41                   19          none   
13675              8                 41                   11          none   

       is_holiday  is_weekend  days_to_holiday  regular_ratio  \
13671           0           1          

In [36]:
#----------------------
# TRAIN-TEST SPLIT
#----------------------

# 2023-2024 for Training
train_data = df[
    df['date'].dt.year.isin([2023, 2024])
].copy()

# 2025 for Testing
test_data = df[
    df['date'].dt.year == 2025
].copy()

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Training rows: 9503
Testing rows: 4745


In [37]:
#----------------------
# DATA VERIFICATION
#----------------------

print(f"--- Data Summary (Post-Pandemic Analysis) ---")
print(f"Training Set (2023-2024): {len(train_data)} rows")
print(f"Testing Set (2025): {len(test_data)} rows")

print("\n--- Training Data Sample ---")
print(train_data.head())

print("\n--- Testing Data Sample ---")
print(test_data.head())

print("\n--- Dataset Sample ---")
print(df.head())

print("\n--- Metadata Verification ---")
df.info()

print(f"\nData range: {df['date'].min()} to {df['date'].max()}")

--- Data Summary (Post-Pandemic Analysis) ---
Training Set (2023-2024): 9503 rows
Testing Set (2025): 4745 rows

--- Training Data Sample ---
            date  state  daily  blood_a  blood_b  blood_ab  blood_o  \
13671 2023-01-01  Johor    346       91       81        18      156   
13672 2023-01-02  Johor    174       43       38         8       85   
13673 2023-01-03  Johor     88       18       38         0       32   
13674 2023-01-04  Johor    122       28       35         8       51   
13675 2023-01-05  Johor     60       20       15         5       20   

       donations_new  donations_regular  donations_irregular    spike_flag  \
13671             67                210                   69  normal_spike   
13672             39                105                   30          none   
13673             21                 48                   19          none   
13674             62                 41                   19          none   
13675              8                 41  

In [38]:
#----------------
# EXPORT
#----------------
import os

# Create the 'data' directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')
    
train_data.to_csv("data/train_data.csv", index=False)
test_data.to_csv("data/test_data.csv", index=False)

print("CSV files exported successfully.")

CSV files exported successfully.
